In [13]:
import pandas as pd
import os

In [14]:
file_path = "../data/raw/AMI AND TAMPA SCORE TRACKING SHEETS.xlsx"

xls = pd.ExcelFile(file_path)
xls.sheet_names

['AMI SCORES (2026)',
 'AMI SCORES (2025)',
 'AMI SCORES (2024)',
 'AMI SCORES (2023)']

In [15]:
dfs = []

for sheet in xls.sheet_names:
    temp = pd.read_excel(file_path, sheet_name=sheet)
    temp["source_sheet"] = sheet
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)

In [16]:
df.columns = df.columns.str.strip()
df.columns.tolist()

['Last Name',
 'First Name',
 'Injury',
 'Date of AMI',
 'LEVEL',
 'SCORE %',
 'TSK 11',
 '*DO NOT RE-SORT THE SHEET IT HAS TO STAY IN ALPHABETICAL ORDER',
 'source_sheet']

In [17]:
df = df.loc[:, ~df.columns.str.contains("DO NOT RE-SORT", case=False, na=False)]

In [18]:
df = df.ffill()

In [19]:
df = df.rename(columns={
    "Last Name": "last_name",
    "First Name": "first_name",
    "Injury": "injury",
    "Date of AMI": "date_ami",
    "LEVEL": "level",
    "SCORE %": "score_pct",
    "TSK 11": "tsk_11"
})

In [20]:
df["date_ami"] = pd.to_datetime(df["date_ami"], format="mixed", errors="coerce")
df["score_pct"] = pd.to_numeric(df["score_pct"], errors="coerce")
df["tsk_11"] = pd.to_numeric(df["tsk_11"], errors="coerce")

In [21]:
acl_df = df[df["injury"].str.upper().eq("ACL")].copy()

In [22]:
analysis_df = acl_df.dropna(subset=["score_pct", "tsk_11"]).copy()

In [23]:
analysis_df.shape
analysis_df.head()
analysis_df.info()
analysis_df.isna().sum()

<class 'pandas.DataFrame'>
Index: 430 entries, 315 to 1877
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   last_name     430 non-null    str           
 1   first_name    430 non-null    str           
 2   injury        430 non-null    str           
 3   date_ami      430 non-null    datetime64[us]
 4   level         430 non-null    object        
 5   score_pct     430 non-null    float64       
 6   tsk_11        430 non-null    float64       
 7   source_sheet  430 non-null    str           
dtypes: datetime64[us](1), float64(2), object(1), str(4)
memory usage: 30.2+ KB


last_name       0
first_name      0
injury          0
date_ami        0
level           0
score_pct       0
tsk_11          0
source_sheet    0
dtype: int64

In [24]:
os.makedirs("../data/processed", exist_ok=True)

analysis_df.to_csv("../data/processed/acl_analysis.csv", index=False)